# Seasonality and Dummy Variables

Seasonal regression adds indicator variables for repeated periods such as quarters or months. For a cycle with L seasons, use L - 1 dummy variables and leave one season as the baseline. Leaving out one season avoids perfect collinearity with the intercept. With quarterly data and Q1 as the baseline, define:

- $Q2_t = 1$ if period $t$ is quarter 2, and 0 otherwise.
- $Q3_t = 1$ if period $t$ is quarter 3, and 0 otherwise.
- $Q4_t = 1$ if period $t$ is quarter 4, and 0 otherwise.

Then a linear trend plus additive seasonal model is

$$y_t = \beta_0 + \beta_1 t + \beta_2 Q2_t + \beta_3 Q3_t + \beta_4 Q4_t + \epsilon_t.$$

Q1 is the baseline season because all three dummy variables are zero in Q1.

In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson
from checks import check_columns, check_no_missing

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True

seasonal_sales = pd.DataFrame({
    'Year':    [1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4],
    'Quarter': [1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4],
    'Time':    list(range(1, 17)),
    'Sales':   [82, 96, 118, 90, 88, 103, 128, 97, 96, 112, 139, 105, 103, 120, 150, 114],
})
print(check_columns(seasonal_sales, ['Year', 'Quarter', 'Time', 'Sales']))
seasonal_sales.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(seasonal_sales['Time'], seasonal_sales['Sales'], marker='o')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Sales')
axes[0].set_title('Teaching example: quarterly equipment sales')
for year, group in seasonal_sales.groupby('Year'):
    axes[1].plot(group['Quarter'], group['Sales'], marker='o', label=f'Year {year}')
axes[1].set_xticks([1, 2, 3, 4])
axes[1].set_xlabel('Quarter')
axes[1].set_ylabel('Sales')
axes[1].set_title('Seasonal display')
axes[1].legend()
plt.tight_layout()


The teaching example shows an increasing linear trend across years and a stable quarterly pattern: Q3 is consistently high, Q1 is consistently low, and the seasonal swing is similar from year to year. That is the additive, or constant, seasonality case.

Because the seasonal swing is roughly constant, we model the original response scale rather than taking a log transformation.


In [ ]:
seasonal_sales = seasonal_sales.assign(
    Q2=(seasonal_sales['Quarter'] == 2).astype(int),
    Q3=(seasonal_sales['Quarter'] == 3).astype(int),
    Q4=(seasonal_sales['Quarter'] == 4).astype(int),
)
seasonal_model = smf.ols('Sales ~ Time + Q2 + Q3 + Q4', data=seasonal_sales).fit()
print(seasonal_model.summary().tables[1])

coefficient_tests = pd.DataFrame({
    'coefficient': seasonal_model.params,
    'p_value': seasonal_model.pvalues,
})
coefficient_tests.round(4)


Coefficient interpretation:

- Intercept: fitted Q1 value when time is 0. It anchors the equation but may not be directly meaningful if time 0 is outside the data.
- Time coefficient: average period-to-period trend after accounting for season.
- Q2, Q3, Q4 coefficients: expected difference from Q1 at the same time.

For statistical importance, use the coefficient table. For each independent variable, the usual individual test is

$$H_0: \beta_j = 0 \quad \text{versus} \quad H_a: \beta_j \ne 0,$$

holding the other predictors in the model fixed. A small p-value, such as below 0.05, is evidence that the predictor contributes to the model after accounting for the other predictors. For a seasonal dummy, that means the season differs from the baseline season at the same time value.


In [ ]:
alpha = 0.05
terms = ['Time', 'Q2', 'Q3', 'Q4']
importance = pd.DataFrame({
    'term': terms,
    'p_value': seasonal_model.pvalues[terms],
    'important_at_0.05': seasonal_model.pvalues[terms] < alpha,
})
importance.round(4)


In [ ]:
future_quarters = pd.DataFrame({
    'Time': [17, 18, 19, 20],
    'Quarter': [1, 2, 3, 4],
})
future_quarters = future_quarters.assign(
    Q2=(future_quarters['Quarter'] == 2).astype(int),
    Q3=(future_quarters['Quarter'] == 3).astype(int),
    Q4=(future_quarters['Quarter'] == 4).astype(int),
)
pred = seasonal_model.get_prediction(future_quarters).summary_frame(alpha=0.05)
forecast_table = future_quarters.join(pred[['mean', 'mean_ci_lower', 'mean_ci_upper', 'obs_ci_lower', 'obs_ci_upper']])
forecast_table.round(2)


The confidence interval estimates the mean response for that future season. The prediction interval estimates a single future observation and is wider because it includes future random error.


## Fitted Equation and Manual Forecasts

Once the coefficients are estimated, write the fitted additive seasonal equation as

$$\hat y_t = b_0 + b_1 t + b_2 Q2_t + b_3 Q3_t + b_4 Q4_t.$$

To calculate a forecast manually, plug in the future time value and the correct dummy values. For Q1, all three dummy variables are 0. For Q2, use $Q2=1, Q3=0, Q4=0$; similarly for Q3 and Q4.


In [ ]:
b = seasonal_model.params

def manual_forecast(row):
    return (
        b['Intercept']
        + b['Time'] * row['Time']
        + b['Q2'] * row['Q2']
        + b['Q3'] * row['Q3']
        + b['Q4'] * row['Q4']
    )

manual_check = future_quarters.iloc[:2].copy()
manual_check['manual_forecast'] = manual_check.apply(manual_forecast, axis=1)
manual_check['model_predict'] = seasonal_model.predict(manual_check)
manual_check.round(2)


Transfer question: if the series had monthly data, how many dummy variables would you include when one month is the baseline?
